# Notebook III-b: Conditional GAN for SPECT Attenuation Correction

Learn to build a conditional adversarial network that transforms non-attenuation-corrected SPECT images into attenuation-corrected ones — no CT scan required.

**What you'll learn:**
- The clinical motivation for CT-free attenuation correction (DeepAC)
- U-Net generator architecture with skip connections
- PatchGAN discriminator
- Combined adversarial + L1 training
- Clinical evaluation metrics (SSIM, PSNR, diagnostic accuracy)

**References:**
- DeepAC concept for SPECT AC (JNM 64/3/472)
- Pix2pix (Isola et al., CVPR 2017)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity, peak_signal_noise_ratio

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## The Attenuation Correction Problem

SPECT (Single Photon Emission Computed Tomography) measures gamma-ray photon emission from radioactive tracers injected into the bloodstream. The photons must travel through tissue to reach the detector, and **tissue absorbs (attenuates) photons** along the way.

**Without attenuation correction:**
- Regions behind thick tissue appear to have reduced tracer uptake
- Creates **apparent perfusion defects** where tissue is thick (e.g., the inferior wall in large patients)
- False positives reduce diagnostic specificity

**Traditional solution:**
1. Acquire a CT scan alongside SPECT
2. Derive an attenuation map from CT Hounsfield units
3. Apply iterative reconstruction with attenuation correction

**Problems with CT-based AC:**
- Extra radiation dose to the patient (~1-3 mSv)
- Spatial misregistration between CT and SPECT (patient motion, breathing)
- Not all SPECT cameras have CT capability

**The DeepAC idea:** Learn the mapping from non-attenuation-corrected images to attenuation-corrected images directly from paired training data, bypassing CT entirely.

## Why Conditional GAN?

**Why not just L1/L2 regression?**
- Naive pixel-wise losses (L1, L2) produce **blurry outputs** because the model averages over multiple plausible outputs
- For medical images, this blurriness can obscure subtle defect boundaries

**Adversarial loss forces realism:**
- A discriminator network learns to distinguish real AC images from generated ones
- The generator must produce outputs that are indistinguishable from real AC images
- This encourages **sharp, realistic textures** rather than blurry averages

**Conditional GAN:**
- The discriminator sees the **input** (non-AC) alongside the output
- This ensures the generated AC image is consistent with the specific input, not just any realistic-looking image

**Combined loss:**
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{adversarial}} + \lambda \cdot \mathcal{L}_{L1}$$

- $\lambda$ is typically **100** (L1 dominates for structural correctness, adversarial adds sharpness)
- L1 preferred over L2 because it produces less blurring

In [ ]:
def generate_paired_spect_data(n_samples=500, image_size=64):
    """Generate synthetic paired non-AC/AC SPECT images.
    
    Simulates the effect of attenuation on myocardial perfusion images.
    AC images: clean perfusion pattern
    Non-AC images: same pattern with attenuation artifacts
    """
    pairs = []
    
    y_grid, x_grid = np.mgrid[-1:1:image_size*1j, -1:1:image_size*1j]
    r = np.sqrt(x_grid**2 + y_grid**2)
    angle = np.arctan2(y_grid, x_grid)
    
    for _ in range(n_samples):
        # Create AC image (ground truth perfusion)
        ac = np.zeros((image_size, image_size), dtype=np.float32)
        
        # Myocardial ring with random rotation and slight deformation
        ring_inner = 0.2 + np.random.uniform(-0.03, 0.03)
        ring_outer = 0.45 + np.random.uniform(-0.03, 0.03)
        ring = (r > ring_inner) & (r < ring_outer)
        
        # Base uptake with some variation
        ac[ring] = 0.8 + np.random.uniform(-0.1, 0.1)
        
        # Random perfusion variations (normal)
        for _ in range(np.random.randint(2, 5)):
            ang_center = np.random.uniform(-np.pi, np.pi)
            ang_width = np.random.uniform(0.3, 0.8)
            mask = ring & (np.abs(angle - ang_center) < ang_width)
            ac[mask] *= np.random.uniform(0.85, 1.15)
        
        # Some samples have real defects
        if np.random.random() < 0.3:
            defect_angle = np.random.uniform(-np.pi, np.pi)
            defect_width = np.random.uniform(0.3, 0.7)
            defect_mask = ring & (np.abs(angle - defect_angle) < defect_width)
            ac[defect_mask] *= np.random.uniform(0.2, 0.5)
        
        # Background
        ac += 0.05
        
        # Add mild noise to AC
        ac += np.random.normal(0, 0.02, ac.shape).astype(np.float32)
        ac = np.clip(ac, 0, 1)
        
        # Create non-AC image (apply synthetic attenuation)
        # Attenuation is stronger through thicker tissue (body center)
        attenuation_map = np.exp(-1.5 * np.maximum(0, 0.6 - r))
        # Directional bias (anterior thinner than posterior)
        attenuation_map *= (1 - 0.3 * np.clip(y_grid, 0, 1))
        
        non_ac = ac * attenuation_map
        
        # More noise in non-AC (lower counts)
        non_ac += np.random.normal(0, 0.04, non_ac.shape).astype(np.float32)
        non_ac = np.clip(non_ac, 0, 1)
        
        pairs.append((non_ac, ac))
    
    return pairs

# Generate data
train_pairs = generate_paired_spect_data(400)
val_pairs = generate_paired_spect_data(100)
print(f"Training pairs: {len(train_pairs)}, Validation pairs: {len(val_pairs)}")

In [ ]:
# Visualize some paired examples
fig, axes = plt.subplots(4, 3, figsize=(10, 12))

for i in range(4):
    non_ac, ac = train_pairs[i]
    diff = ac - non_ac
    
    axes[i, 0].imshow(non_ac, cmap='hot', vmin=0, vmax=1)
    axes[i, 0].set_title('Non-AC (input)' if i == 0 else '')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(ac, cmap='hot', vmin=0, vmax=1)
    axes[i, 1].set_title('AC (target)' if i == 0 else '')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(diff, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[i, 2].set_title('Difference (AC - non-AC)' if i == 0 else '')
    axes[i, 2].axis('off')

plt.suptitle('Paired Training Data: Non-AC vs AC SPECT Images', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

print("Notice how the non-AC images show reduced intensity in the inferior/posterior")
print("regions due to attenuation through thicker tissue.")

In [ ]:
class SPECTDataset(Dataset):
    """Dataset for paired non-AC / AC SPECT images."""
    
    def __init__(self, pairs):
        self.pairs = pairs
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        non_ac, ac = self.pairs[idx]
        # Add channel dimension: (1, H, W)
        non_ac_tensor = torch.FloatTensor(non_ac).unsqueeze(0)
        ac_tensor = torch.FloatTensor(ac).unsqueeze(0)
        return non_ac_tensor, ac_tensor

# Quick sanity check
ds = SPECTDataset(train_pairs)
sample_in, sample_out = ds[0]
print(f"Input shape: {sample_in.shape}, Output shape: {sample_out.shape}")
print(f"Input range: [{sample_in.min():.3f}, {sample_in.max():.3f}]")
print(f"Output range: [{sample_out.min():.3f}, {sample_out.max():.3f}]")

## Generator: U-Net Architecture

The generator follows the **U-Net** architecture, originally designed for biomedical image segmentation and adopted by pix2pix for image-to-image translation.

**Encoder path** (downsampling):
- `Conv2d(stride=2)` &rarr; `InstanceNorm2d` &rarr; `LeakyReLU(0.2)`
- Progressively reduces spatial dimensions while increasing feature channels
- Captures increasingly abstract representations

**Bottleneck:**
- Deepest, most compressed representation
- Captures global context of the image

**Decoder path** (upsampling):
- `ConvTranspose2d(stride=2)` &rarr; `InstanceNorm2d` &rarr; `ReLU`
- **Skip connections**: concatenate encoder features at each level
- Skip connections preserve fine spatial detail that would otherwise be lost in the bottleneck

**Final layer:** `Conv` &rarr; `Sigmoid` to map output to [0, 1]

**Note:** We use 2D convolutions for this tutorial. For volumetric SPECT data, replace `Conv2d` with `Conv3d` — the architecture is otherwise identical.

```
Input (1, 64, 64)
  |--- enc1 ---> (32, 32, 32) ----skip----|
  |--- enc2 ---> (64, 16, 16) ----skip----|---|
  |--- enc3 ---> (128, 8, 8) -----skip----|---|---|
  |--- enc4 ---> (256, 4, 4) -----skip----|---|---|---|
  |--- bottleneck -> (512, 2, 2)           |   |   |   |
  |--- dec4 ---> (256, 4, 4) + skip -------|---|---|---|
  |--- dec3 ---> (128, 8, 8) + skip -------|---|---|
  |--- dec2 ---> (64, 16, 16) + skip ------|---|
  |--- dec1 ---> (32, 32, 32) + skip ------|
  |--- final --> (1, 64, 64)
```

In [ ]:
class UNetGenerator(nn.Module):
    """U-Net generator with skip connections for image-to-image translation."""
    
    def __init__(self, in_channels=1, out_channels=1, features=32):
        super().__init__()
        
        # Encoder
        self.enc1 = self._encoder_block(in_channels, features)       # 64->32
        self.enc2 = self._encoder_block(features, features * 2)      # 32->16
        self.enc3 = self._encoder_block(features * 2, features * 4)  # 16->8
        self.enc4 = self._encoder_block(features * 4, features * 8)  # 8->4
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(features * 8, features * 16, 4, 2, 1),         # 4->2
            nn.LeakyReLU(0.2, inplace=True),
        )
        
        # Decoder (with skip connections -> double input channels)
        self.dec4 = self._decoder_block(features * 16, features * 8)  # 2->4
        self.dec3 = self._decoder_block(features * 16, features * 4)  # 4->8  (8*2=16 from skip)
        self.dec2 = self._decoder_block(features * 8, features * 2)   # 8->16
        self.dec1 = self._decoder_block(features * 4, features)       # 16->32
        
        # Final
        self.final = nn.Sequential(
            nn.ConvTranspose2d(features * 2, out_channels, 4, 2, 1),  # 32->64
            nn.Sigmoid(),
        )
    
    def _encoder_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_c),
            nn.LeakyReLU(0.2, inplace=True),
        )
    
    def _decoder_block(self, in_c, out_c):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        # Encode
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        
        # Bottleneck
        b = self.bottleneck(e4)
        
        # Decode with skip connections
        d4 = self.dec4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d3 = self.dec3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d2 = self.dec2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d1 = self.dec1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        
        return self.final(d1)

# Verify shapes
_test_g = UNetGenerator()
_test_input = torch.randn(1, 1, 64, 64)
_test_output = _test_g(_test_input)
print(f"Generator: {_test_input.shape} -> {_test_output.shape}")
del _test_g, _test_input, _test_output

## Discriminator: PatchGAN

Instead of classifying the **entire image** as real or fake, the PatchGAN discriminator classifies **overlapping patches**.

**Key properties:**
- Output is a **grid of real/fake probabilities** (e.g., 6x6 grid for a 64x64 input)
- Each output neuron has a receptive field of roughly 70x70 pixels
- Captures **local texture statistics** rather than global structure
- Global structure is already enforced by the L1 loss

**Conditional discriminator:**
- Input: concatenation of non-AC image and target/generated AC image along the channel dimension
- The discriminator learns whether the *pair* is consistent
- This prevents the generator from producing a generic realistic AC image that doesn't match the specific input

In [ ]:
class PatchGANDiscriminator(nn.Module):
    """PatchGAN discriminator -- classifies overlapping patches as real/fake."""
    
    def __init__(self, in_channels=2):  # 2 = input + target/generated
        super().__init__()
        
        self.model = nn.Sequential(
            # No normalization on first layer
            nn.Conv2d(in_channels, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(256, 1, 4, 1, 1),  # Output: patch-level real/fake
        )
    
    def forward(self, input_img, target_img):
        x = torch.cat([input_img, target_img], dim=1)
        return self.model(x)

# Verify shapes
_test_d = PatchGANDiscriminator()
_test_in = torch.randn(1, 1, 64, 64)
_test_tgt = torch.randn(1, 1, 64, 64)
_test_out = _test_d(_test_in, _test_tgt)
print(f"Discriminator: input {_test_in.shape} + target {_test_tgt.shape} -> {_test_out.shape}")
print(f"Each output value represents a patch-level real/fake prediction")
del _test_d, _test_in, _test_tgt, _test_out

In [ ]:
# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

generator = UNetGenerator().to(device)
discriminator = PatchGANDiscriminator().to(device)

opt_g = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

criterion_adv = nn.MSELoss()   # LSGAN loss (more stable than BCE)
criterion_l1 = nn.L1Loss()
lambda_l1 = 100

train_loader = DataLoader(
    SPECTDataset(train_pairs), batch_size=16, shuffle=True
)
val_loader = DataLoader(
    SPECTDataset(val_pairs), batch_size=16, shuffle=False
)

n_params_g = sum(p.numel() for p in generator.parameters())
n_params_d = sum(p.numel() for p in discriminator.parameters())
print(f"Generator parameters:     {n_params_g:,}")
print(f"Discriminator parameters: {n_params_d:,}")
print(f"Training batches per epoch: {len(train_loader)}")

## Training Loop

GANs are trained with **alternating optimization** — each iteration has two phases:

### Phase 1: Train Discriminator
The discriminator learns to distinguish real from generated pairs:
1. **Real pair** (non-AC, real AC) &rarr; discriminator should output **1** (real)
2. **Fake pair** (non-AC, G(non-AC)) &rarr; discriminator should output **0** (fake)
3. Loss = MSE(D(real_pair), 1) + MSE(D(fake_pair), 0)

### Phase 2: Train Generator
The generator learns to fool the discriminator AND reconstruct the target:
1. Generate fake AC: `fake_ac = G(non_ac)`
2. **Adversarial loss**: MSE(D(non_ac, fake_ac), 1) &mdash; fool D into saying "real"
3. **L1 loss**: |fake_ac - real_ac| &mdash; pixel-wise reconstruction
4. Total: L_adv + 100 * L_L1

We use **LSGAN** (Least Squares GAN) instead of the original cross-entropy GAN loss. LSGAN uses MSE instead of BCE, which provides more stable gradients and avoids vanishing gradient problems.

In [ ]:
# Training loop
num_epochs = 50

history = {
    'g_loss': [], 'd_loss': [], 'g_adv': [], 'g_l1': [],
    'val_l1': []
}

for epoch in range(num_epochs):
    generator.train()
    discriminator.train()
    
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    epoch_g_adv = 0.0
    epoch_g_l1 = 0.0
    n_batches = 0
    
    for non_ac, ac in train_loader:
        non_ac = non_ac.to(device)
        ac = ac.to(device)
        batch_size = non_ac.size(0)
        
        # Generate fake AC images
        fake_ac = generator(non_ac)
        
        # ---- Train Discriminator ----
        opt_d.zero_grad()
        
        # Real pair
        pred_real = discriminator(non_ac, ac)
        label_real = torch.ones_like(pred_real)
        loss_d_real = criterion_adv(pred_real, label_real)
        
        # Fake pair (detach generator output)
        pred_fake = discriminator(non_ac, fake_ac.detach())
        label_fake = torch.zeros_like(pred_fake)
        loss_d_fake = criterion_adv(pred_fake, label_fake)
        
        loss_d = (loss_d_real + loss_d_fake) * 0.5
        loss_d.backward()
        opt_d.step()
        
        # ---- Train Generator ----
        opt_g.zero_grad()
        
        # Adversarial loss: fool discriminator
        pred_fake_for_g = discriminator(non_ac, fake_ac)
        loss_g_adv = criterion_adv(pred_fake_for_g, label_real)  # Want D to say "real"
        
        # L1 reconstruction loss
        loss_g_l1 = criterion_l1(fake_ac, ac)
        
        loss_g = loss_g_adv + lambda_l1 * loss_g_l1
        loss_g.backward()
        opt_g.step()
        
        epoch_g_loss += loss_g.item()
        epoch_d_loss += loss_d.item()
        epoch_g_adv += loss_g_adv.item()
        epoch_g_l1 += loss_g_l1.item()
        n_batches += 1
    
    # Average over batches
    history['g_loss'].append(epoch_g_loss / n_batches)
    history['d_loss'].append(epoch_d_loss / n_batches)
    history['g_adv'].append(epoch_g_adv / n_batches)
    history['g_l1'].append(epoch_g_l1 / n_batches)
    
    # Validation L1
    generator.eval()
    val_l1 = 0.0
    val_batches = 0
    with torch.no_grad():
        for non_ac, ac in val_loader:
            non_ac, ac = non_ac.to(device), ac.to(device)
            fake_ac = generator(non_ac)
            val_l1 += criterion_l1(fake_ac, ac).item()
            val_batches += 1
    history['val_l1'].append(val_l1 / val_batches)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:3d}/{num_epochs}]  "
              f"D_loss: {history['d_loss'][-1]:.4f}  "
              f"G_loss: {history['g_loss'][-1]:.4f}  "
              f"G_adv: {history['g_adv'][-1]:.4f}  "
              f"G_L1: {history['g_l1'][-1]:.4f}  "
              f"Val_L1: {history['val_l1'][-1]:.4f}")

print("\nTraining complete.")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, num_epochs + 1)

# Discriminator loss
axes[0].plot(epochs, history['d_loss'], 'b-', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Discriminator Loss')
axes[0].grid(True, alpha=0.3)

# Generator losses
axes[1].plot(epochs, history['g_adv'], 'r-', label='Adversarial', linewidth=1.5)
axes[1].plot(epochs, history['g_loss'], 'k--', label='Total (adv + 100*L1)', linewidth=1.0, alpha=0.7)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Generator Losses')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# L1 loss (train + val)
axes[2].plot(epochs, history['g_l1'], 'g-', label='Train L1', linewidth=1.5)
axes[2].plot(epochs, history['val_l1'], 'orange', label='Val L1', linewidth=1.5)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('L1 Loss')
axes[2].set_title('Reconstruction Loss (L1)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Look for:")
print("  - D_loss should stabilize around 0.25 (equilibrium)")
print("  - G_L1 should decrease steadily")
print("  - Val_L1 should track Train_L1 (no overfitting with small model)")

In [ ]:
# Evaluate on validation set: compute SSIM, PSNR, NMSE per sample
generator.eval()

val_results = []  # (non_ac, generated, ac, ssim, psnr, nmse)

with torch.no_grad():
    for non_ac, ac in val_loader:
        non_ac, ac = non_ac.to(device), ac.to(device)
        generated = generator(non_ac)
        
        for i in range(non_ac.size(0)):
            gen_np = generated[i, 0].cpu().numpy()
            ac_np = ac[i, 0].cpu().numpy()
            nac_np = non_ac[i, 0].cpu().numpy()
            
            ssim_val = structural_similarity(ac_np, gen_np, data_range=1.0)
            psnr_val = peak_signal_noise_ratio(ac_np, gen_np, data_range=1.0)
            nmse_val = np.mean((gen_np - ac_np)**2) / np.mean(ac_np**2)
            
            val_results.append((nac_np, gen_np, ac_np, ssim_val, psnr_val, nmse_val))

print(f"Evaluated {len(val_results)} validation samples")

In [ ]:
# Visual comparison grid: 4 validation examples
fig, axes = plt.subplots(4, 4, figsize=(16, 14))

# Pick 4 examples spread across the validation set
indices = [0, 25, 50, 75]

for row, idx in enumerate(indices):
    nac_np, gen_np, ac_np, ssim_val, psnr_val, nmse_val = val_results[idx]
    
    # Non-AC input
    axes[row, 0].imshow(nac_np, cmap='hot', vmin=0, vmax=1)
    axes[row, 0].set_title('Non-AC (input)' if row == 0 else '')
    axes[row, 0].axis('off')
    
    # Generated AC
    axes[row, 1].imshow(gen_np, cmap='hot', vmin=0, vmax=1)
    title = 'Generated AC' if row == 0 else ''
    axes[row, 1].set_title(title)
    axes[row, 1].axis('off')
    
    # Ground truth AC
    axes[row, 2].imshow(ac_np, cmap='hot', vmin=0, vmax=1)
    axes[row, 2].set_title('Ground Truth AC' if row == 0 else '')
    axes[row, 2].axis('off')
    
    # Absolute error
    error = np.abs(gen_np - ac_np)
    im = axes[row, 3].imshow(error, cmap='magma', vmin=0, vmax=0.3)
    axes[row, 3].set_title('|Error|' if row == 0 else '')
    axes[row, 3].axis('off')
    
    # Annotate with metrics
    axes[row, 1].text(
        0.5, -0.08, f"SSIM={ssim_val:.3f}  PSNR={psnr_val:.1f}dB",
        transform=axes[row, 1].transAxes, ha='center', fontsize=9,
        color='darkblue', fontweight='bold'
    )

plt.suptitle('Conditional GAN Results: Non-AC -> Generated AC vs Ground Truth',
             fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

## Clinical Evaluation Discussion

Image quality metrics like SSIM and PSNR are **necessary but not sufficient** for clinical deployment.

### What image metrics tell us
- SSIM > 0.9: structural similarity is high
- PSNR > 25 dB: pixel-level error is low
- These confirm the model is learning the attenuation correction mapping

### What clinical validation requires
- **Diagnostic accuracy**: Does the predicted-AC image lead to the *same clinical diagnosis* as the true-AC image?
- **Total Perfusion Deficit (TPD) agreement**: Quantitative perfusion scores should match between predicted-AC and true-AC
- **AUC for detecting obstructive CAD**: Can the predicted-AC images detect coronary artery disease as accurately as true-AC?
- **Normalcy rates**: In patients with low pre-test likelihood, the predicted-AC images should yield normal results at the same rate as true-AC

### Risks specific to GANs in medical imaging
- **Mode collapse**: Generator produces limited variety of outputs, potentially missing rare but clinically important patterns
- **Hallucination**: GAN may invent features not present in the input (e.g., "correcting" a real defect that is actually present)
- **Distribution shift**: Model trained on one scanner/protocol may fail on another

### Regulatory pathway
- FDA 510(k) or De Novo clearance required for clinical deployment
- Multi-site validation studies needed
- Must demonstrate non-inferiority to CT-based AC
- Post-market surveillance for failure modes

In [ ]:
# Quantitative metrics summary across all validation samples
ssim_scores = [r[3] for r in val_results]
psnr_scores = [r[4] for r in val_results]
nmse_scores = [r[5] for r in val_results]

print(f"Validation Metrics (n={len(ssim_scores)})")
print(f"{'':4s}{'Metric':>8s}{'Mean':>12s}{'Std':>12s}{'Min':>12s}{'Max':>12s}")
print(f"{'':4s}{'-'*56}")
print(f"{'':4s}{'SSIM':>8s}{np.mean(ssim_scores):12.4f}{np.std(ssim_scores):12.4f}"
      f"{np.min(ssim_scores):12.4f}{np.max(ssim_scores):12.4f}")
print(f"{'':4s}{'PSNR':>8s}{np.mean(psnr_scores):12.2f}{np.std(psnr_scores):12.2f}"
      f"{np.min(psnr_scores):12.2f}{np.max(psnr_scores):12.2f}  dB")
print(f"{'':4s}{'NMSE':>8s}{np.mean(nmse_scores):12.6f}{np.std(nmse_scores):12.6f}"
      f"{np.min(nmse_scores):12.6f}{np.max(nmse_scores):12.6f}")

# Also compute baseline metrics (non-AC vs AC, without any correction)
baseline_ssim = []
baseline_psnr = []
for r in val_results:
    nac_np, _, ac_np = r[0], r[1], r[2]
    baseline_ssim.append(structural_similarity(ac_np, nac_np, data_range=1.0))
    baseline_psnr.append(peak_signal_noise_ratio(ac_np, nac_np, data_range=1.0))

print(f"\nBaseline (no correction, non-AC vs AC):")
print(f"    SSIM: {np.mean(baseline_ssim):.4f} +/- {np.std(baseline_ssim):.4f}")
print(f"    PSNR: {np.mean(baseline_psnr):.2f} +/- {np.std(baseline_psnr):.2f} dB")
print(f"\nImprovement from cGAN:")
print(f"    SSIM: {np.mean(baseline_ssim):.4f} -> {np.mean(ssim_scores):.4f} "
      f"(+{np.mean(ssim_scores) - np.mean(baseline_ssim):.4f})")
print(f"    PSNR: {np.mean(baseline_psnr):.2f} -> {np.mean(psnr_scores):.2f} "
      f"(+{np.mean(psnr_scores) - np.mean(baseline_psnr):.2f} dB)")

## Summary and Next Steps

### What we built
- A **conditional GAN** (pix2pix-style) for SPECT attenuation correction
- **U-Net generator** with skip connections preserves spatial detail
- **PatchGAN discriminator** enforces local texture realism
- **Combined loss** (adversarial + L1) balances realism and accuracy

### Key takeaways
- Conditional GANs are powerful for **paired image-to-image translation** tasks
- For SPECT AC, they can potentially **replace CT scans** (the DeepAC concept), eliminating extra radiation dose and misregistration artifacts
- The key challenge is **clinical validation** beyond image quality metrics
- GAN-specific risks (mode collapse, hallucination) require careful mitigation in medical applications

### Next notebook
**Notebook III-c** explores **diffusion models** as an alternative generative approach:
- No mode collapse (training is stable, non-adversarial)
- Built-in **uncertainty quantification** via multiple sampling
- State-of-the-art image quality
- Trade-off: slower inference (multiple denoising steps)